# Field Strength Operators Walkthrough

This notebook mirrors the user interface of `list_lagrangians.ipynb` and focuses on the
general `FieldStrength(...)` compiler. Every `FieldStrength` factor is expanded into its
explicit local pieces

$$F^a_{\mu\nu} = \partial_\mu A^a_\nu - \partial_\nu A^a_\mu + g\, f^{abc} A^b_\mu A^c_\nu$$

(abelian groups drop the structure-constant piece), and the resulting product of fields,
derivatives, couplings and structure constants is compiled by the ordinary local-monomial
machinery. As a result, arbitrary `F^n` operators compile without any dedicated
cubic/quartic handlers.

## Setup
### Imports & helpers


In [1]:
import re
import sys
from pathlib import Path
from collections import Counter
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    CovD,
    Field,
    FieldStrength,
    GaugeGroup,
    GaugeRepresentation,
    Model,
    StructureConstant,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = sum(source_terms[1:], source_terms[0]) if len(source_terms) > 1 else source_terms[0]
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)



def show_expansion(title, model):
    """Show how a FieldStrength operator distributes into local interaction terms.

    For higher F^n operators the full set of Feynman rules can involve very
    high-leg vertices, so this helper reports the declared Lagrangian together
    with the number of compiled local terms and their per-vertex multiplicity.
    """
    source_terms = model.lagrangian_decl.source_terms
    lagrangian_source = sum(source_terms[1:], source_terms[0]) if len(source_terms) > 1 else source_terms[0]
    compiled = model.lagrangian()
    counts = Counter(len(term.fields) for term in compiled.terms)
    print("==========")
    print(title)
    print("Lagrangian:", clean(lagrangian_source))
    print(f"Compiled into {len(compiled.terms)} local interaction term(s)")
    for n_fields in sorted(counts):
        print(f"  {n_fields}-point pieces: {counts[n_fields]}")
    print()


### Symbols, fields, and gauge groups

The same `U1QED`, `SU3C`, and `SU2L` declarations used in `list_lagrangians.ipynb`.

In [2]:
mu, nu, rho, sigma = S("mu"), S("nu"), S("rho"), S("sigma")
a, b, c = S("a"), S("b"), S("c")
aC, aW = S("aC"), S("aW")
eQED, gS, g2 = S("eQED"), S("gS"), S("g2")

Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))

COLOR_FUND_REP = GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fund")
WEAK_DOUBLET_REP = GaugeRepresentation(index=WEAK_FUND_INDEX, generator_builder=weak_gauge_generator, name="doublet")

U1QED = GaugeGroup(name="U1QED", abelian=True, coupling=eQED, gauge_boson=Photon, charge="Q")
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={}, ghost_of=None, flavor_index=None, class_members=()), ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', dimension=None, is_flavor=False, prefix='a')), kind='vector', statistics='boson', symbol

## Abelian field strength: photon kinetic term

An abelian `FieldStrength(U1, mu, nu)` takes **no** adjoint index.

In [3]:
photon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu),
)
show_expansion("Abelian F^2 (photon)", photon_kinetic_model)
show_model(photon_kinetic_model)


Abelian F^2 (photon)
Lagrangian: -1/4 * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu)
Compiled into 4 local interaction term(s)
  2-point pieces: 4

Lagrangian
-1/4 * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu)

Feynman Rules
1 vertex signature(s)

Vertex: ('A', 'A')
Rule: -1𝑖*pcomp(q1,mu2)*pcomp(q2,mu1)+1𝑖*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)



## Non-abelian field strength

A non-abelian `FieldStrength(SU3C, mu, nu, a)` **requires** an explicit adjoint index `a`.


In [4]:
gluon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC),
)
show_expansion("Non-abelian F^2 (gluon)", gluon_kinetic_model)
show_model(gluon_kinetic_model)

Non-abelian F^2 (gluon)
Lagrangian: -1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)
Compiled into 9 local interaction term(s)
  2-point pieces: 4
  3-point pieces: 4
  4-point pieces: 1

Lagrangian
-1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)

Feynman Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu2)*pcomp(q2,mu1)

Vertex: ('G', 'G', 'G')
Rule: gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu3)-gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu3)-gS*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)+gS*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q3,mu2)+gS*g(mink(4, mu2),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu1)-gS*g(mink(4, mu2),mink(4, mu3))*

## Cubic color operator: `f^{abc} F^a F^b F^c`

In [9]:
f_f3_model = Model(
    S("c3")
    * StructureConstant(a, b, c)
    * FieldStrength(SU3C, mu, nu, a)
    * FieldStrength(SU3C, nu, rho, b)
    * FieldStrength(SU3C, rho, mu, c),
)
show_expansion("f^{abc} F^a F^b F^c", f_f3_model)

show_model(f_f3_model)


f^{abc} F^a F^b F^c
Lagrangian: c3 * StructureConstant(a, b, c) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, nu, rho, b) * FieldStrength(SU3C, rho, mu, c)
Compiled into 27 local interaction term(s)
  3-point pieces: 8
  4-point pieces: 12
  5-point pieces: 6
  6-point pieces: 1

Lagrangian
c3 * StructureConstant(a, b, c) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, nu, rho, b) * FieldStrength(SU3C, rho, mu, c)

Feynman Rules
4 vertex signature(s)

Vertex: ('G', 'G', 'G')
Rule: 6*c3*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)*pcomp(q2,mu3)*pcomp(q3,mu1)-6*c3*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu3)*pcomp(q2,mu1)*pcomp(q3,mu2)-4*c3*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu1_int)*pcomp(q2,mu3)*pcomp(q3,mu1_int)-2*c3*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2_int)*pcomp(q2,mu3)*pcomp(q3,mu2_int)+4*c3*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q

## Quartic operator: `(F^a F^a)(F^b F^b)`

`F^4` 3^4 = 81 local monomials

In [6]:
f4_model = Model(
    S("lam")
    * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, mu, nu, a)
    * FieldStrength(SU3C, rho, sigma, b) * FieldStrength(SU3C, rho, sigma, b),
)
show_expansion("(F^a F^a)(F^b F^b)", f4_model)


(F^a F^a)(F^b F^b)
Lagrangian: lam * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, rho, sigma, b) * FieldStrength(SU3C, rho, sigma, b)
Compiled into 81 local interaction term(s)
  4-point pieces: 16
  5-point pieces: 32
  6-point pieces: 24
  7-point pieces: 8
  8-point pieces: 1



## Mixed gauge groups: `F_{SU3}^2 * F_{SU2}^2`


In [10]:
mixed_model = Model(
    S("kappa")
    * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)
    * FieldStrength(SU2L, rho, sigma, aW) * FieldStrength(SU2L, rho, sigma, aW),
    gauge_groups=(SU3C, SU2L),
    fields=(Gluon, W),
)
show_expansion("F_{SU3}^2 * F_{SU2}^2", mixed_model)
show_model(mixed_model)


F_{SU3}^2 * F_{SU2}^2
Lagrangian: kappa * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU2L, rho, sigma, aW) * FieldStrength(SU2L, rho, sigma, aW)
Compiled into 81 local interaction term(s)
  4-point pieces: 16
  5-point pieces: 32
  6-point pieces: 24
  7-point pieces: 8
  8-point pieces: 1

Lagrangian
kappa * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU2L, rho, sigma, aW) * FieldStrength(SU2L, rho, sigma, aW)

Feynman Rules
9 vertex signature(s)

Vertex: ('G', 'G', 'W', 'W')
Rule: 16𝑖*kappa*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*g(mink(4, mu3),mink(4, mu4))*g(coad(3, aw3),coad(3, aw4))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)*pcomp(q3,mu2_int)*pcomp(q4,mu2_int)-16𝑖*kappa*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*g(coad(3, aw3),coad(3, aw4))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)*pcomp(q3,mu4)*pcomp(q4,mu3)-16𝑖*kappa*g(coad(8, a1),coad(8, a2))*g(mink(4, mu3),mink(4, mu4))*g(coad(3, aw3)

## Strict-mode errors

The compiler enforces explicit adjoint indices and color-singlet contraction:

- a non-abelian field strength **must** carry an adjoint index,
- an abelian field strength **must not**,
- every adjoint index must be contracted (no open color indices).

In [8]:
def show_error(title, build):
    print("==========")
    print(title)
    try:
        build()
        print("  (no error raised)")
    except ValueError as exc:
        print("  ValueError:", clean(exc))
    print()


show_error(
    "Non-abelian field strength without an adjoint index",
    lambda: Model(
        -(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu) * FieldStrength(SU3C, mu, nu),
    ).lagrangian(),
)

show_error(
    "Abelian field strength with an adjoint index",
    lambda: Model(
        -(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu, a) * FieldStrength(U1QED, mu, nu, a),
    ).lagrangian(),
)

show_error(
    "Open (uncontracted) adjoint color index",
    lambda: Model(
        S("c") * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, rho, sigma, b),
    ).lagrangian(),
)


Non-abelian field strength without an adjoint index
  ValueError: FieldStrength compilation: non-abelian gauge group 'SU3C' field strength requires an explicit adjoint index, e.g. FieldStrength(group, mu, nu, a).

Abelian field strength with an adjoint index
  ValueError: FieldStrength compilation: abelian gauge group 'U1QED' field strength does not take an adjoint index; write FieldStrength(group, mu, nu).

Open (uncontracted) adjoint color index
  ValueError: FieldStrength monomial has open (uncontracted) adjoint index/indices ['python::{}::a', 'python::{}::b']; the Lagrangian term must be a gauge singlet. Contract them against another field strength or a StructureConstant(a, b, c).

